# RubricGraph: Rule-Driven Answer Grading with LangGraph

This notebook implements an auditable answer-grading workflow in LangGraph.

**Core idea:** the grading policy lives in a replaceable plaintext file (`grading_rules.txt`). The Python application does not hard-code subject-specific marking criteria.

Workflow:

```text
rules.txt
   │
   ▼
parse rubric
   │
   ▼
analyze student answer
   │
   ▼
fan-out: evaluate every criterion independently
   │
   ▼
aggregate deterministic score
   │
   ▼
verify grade
   │
   ├── valid ───────────────► final report
   │
   └── invalid ─────────────► regrade
```

The notebook also demonstrates:

- Pydantic structured output
- LangGraph state
- parallel criterion evaluation with `Send`
- deterministic score aggregation
- verification and conditional routing
- a bounded regrading loop
- an audit-friendly JSON result

## 1. Install dependencies

This version uses **Ollama locally**, so no OpenAI API key is required.

Install Ollama separately, start the Ollama server, and pull a model. For example:

```bash
ollama serve
ollama pull llama3.2
```

Then run the installation cell below.

The notebook uses LangGraph with `langchain-ollama`. You can change the model with:

```bash
export RUBRICGRAPH_MODEL=llama3.2
```

or by changing `MODEL_NAME` in the notebook.


In [ ]:
%pip install -U langgraph langchain langchain-openai pydantic

## 2. Imports and configuration

In [ ]:
import os
import json
from pathlib import Path
from typing import TypedDict, Annotated, Literal

from pydantic import BaseModel, Field
from langchain_ollama import ChatOllama
from langgraph.graph import StateGraph, START, END
from langgraph.types import Send

In [ ]:
# Ollama configuration.
# Make sure Ollama is running and the model has been pulled, e.g.:
#   ollama serve
#   ollama pull llama3.2

MODEL_NAME = os.getenv("RUBRICGRAPH_MODEL", "llama3.2")
OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434")

llm = ChatOllama(
    model=MODEL_NAME,
    temperature=0,
    base_url=OLLAMA_BASE_URL,
)

print("Ollama model:", MODEL_NAME)
print("Ollama URL:", OLLAMA_BASE_URL)


## 3. Create the plaintext grading policy

This file is intentionally external to the Python grading logic.

In a real deployment, a teacher or administrator could provide a different `.txt` file without changing the graph implementation.

In [ ]:
rules_text = '''
GRADING RULES

Question:
Explain the difference between a process and a thread.

Maximum marks: 10

Criteria:

1. Process definition
   - Must explain that a process is an independent program in execution.
   - Maximum: 2 marks.

2. Thread definition
   - Must explain that a thread is an execution unit within a process.
   - Maximum: 2 marks.

3. Address space
   - Processes have separate address spaces.
   - Threads belonging to the same process share the process address space.
   - Maximum: 2 marks.

4. Resource sharing
   - Threads share resources such as code, data and open files.
   - Maximum: 1 mark.

5. Context switching
   - Process switching generally has greater overhead than thread switching.
   - Maximum: 1 mark.

6. Comparison
   - Student should explicitly distinguish process and thread using at least
     two meaningful differences.
   - Maximum: 2 marks.

Rules:

- Do not award marks merely because a keyword appears.
- Award marks based on semantic correctness.
- Partially correct explanations may receive partial marks.
- Factually incorrect statements must not receive credit.
- Do not penalize grammar unless it makes the technical meaning unclear.
- Do not require wording identical to the reference answer.
- Do not infer knowledge that is not expressed in the answer.
- Maximum score is 10.
'''.strip()

Path("grading_rules.txt").write_text(rules_text, encoding="utf-8")

print(Path("grading_rules.txt").resolve())
print(rules_text)

## 4. Example question, reference answer, and student answer

The reference answer is contextual evidence. The rubric remains authoritative for the score.

In [ ]:
question = "Explain the difference between a process and a thread."

reference_answer = '''
A process is an independent program in execution with its own virtual address
space and resources. A thread is an execution unit within a process. Threads
belonging to the same process share its address space and resources such as
code, data, and open files. Context switching between processes is generally
more expensive than switching between threads of the same process.
'''.strip()

student_answer = '''
A process is a program that is running. A thread is a smaller process inside
a process. Processes have their own memory while threads share memory. Threads
are faster to switch between than processes.
'''.strip()

print("QUESTION:\n", question)
print("\nREFERENCE ANSWER:\n", reference_answer)
print("\nSTUDENT ANSWER:\n", student_answer)

## 5. Structured data models

The LLM is used to interpret the natural-language rubric, but the resulting grading objects are strongly typed.

This prevents the rest of the application from depending on arbitrary model-generated JSON.

In [ ]:
class Criterion(BaseModel):
    id: int
    name: str
    requirements: list[str]
    max_marks: float
    partial_credit_allowed: bool = True


class Rubric(BaseModel):
    question: str
    maximum_marks: float
    criteria: list[Criterion]
    global_rules: list[str]


class Claim(BaseModel):
    text: str
    concept: str
    assessment: Literal["correct", "incorrect", "partial", "uncertain"]


class AnswerAnalysis(BaseModel):
    claims: list[Claim]
    key_concepts_present: list[str]
    key_concepts_missing: list[str]


class EvidenceMapping(BaseModel):
    criterion_id: int
    criterion_name: str
    supporting_answer_parts: list[str]
    represented_requirements: list[str]
    evidence_status: Literal[
        "sufficient", "partial", "incorrect", "absent", "uncertain"
    ]
    explanation: str


class CriterionResult(BaseModel):
    criterion_id: int
    criterion_name: str
    awarded_marks: float
    maximum_marks: float
    evidence: EvidenceMapping
    missing_requirements: list[str]
    reasoning: str


class VerificationResult(BaseModel):
    valid: bool
    issues: list[str]
    requires_regrading: bool


class FinalReport(BaseModel):
    total_score: float
    maximum_score: float
    percentage: float
    criterion_results: list[CriterionResult]
    overall_feedback: str


## 6. Parse the plaintext rules into a structured rubric

In [ ]:
rubric_llm = llm.with_structured_output(Rubric)

rubric_prompt = f'''
You are a rubric parser.

Convert the following plaintext grading policy into the provided structured
Rubric schema.

Important:
- Preserve every criterion.
- Preserve maximum marks exactly.
- Preserve global grading rules.
- Do not invent criteria.
- The plaintext policy is authoritative.

PLAINTEXT POLICY:
-----------------
{rules_text}
-----------------
'''

rubric = rubric_llm.invoke(rubric_prompt)

print(rubric.model_dump_json(indent=2))

## 7. Analyze the student's answer

This node extracts claims and concepts before criterion-level grading.

It does **not** assign marks.

In [ ]:
analysis_llm = llm.with_structured_output(AnswerAnalysis)

analysis_prompt = f'''
Analyze the student's answer without assigning a numerical grade.

Question:
{question}

Reference answer:
{reference_answer}

Student answer:
{student_answer}

Identify:
1. claims made by the student,
2. the concept associated with each claim,
3. whether each claim is correct, incorrect, partial, or uncertain,
4. important concepts present,
5. important concepts missing.

Do not infer knowledge that is not expressed.
'''

answer_analysis = analysis_llm.invoke(analysis_prompt)

print(answer_analysis.model_dump_json(indent=2))

## 8. Define LangGraph state

The `criterion_results` field uses a reducer so multiple criterion-evaluation branches can write their results back into the shared graph state.

`Send` will be used to fan out one grading task per criterion.

In [ ]:
def append_results(
    existing: list[CriterionResult] | None,
    new: list[CriterionResult] | CriterionResult | None,
):
    existing = existing or []
    if new is None:
        return existing
    if isinstance(new, CriterionResult):
        return existing + [new]
    return existing + new


class GradingState(TypedDict, total=False):
    rules_text: str
    rubric: Rubric
    question: str
    reference_answer: str
    student_answer: str

    answer_analysis: AnswerAnalysis

    # Fan-out branches append to this list.
    criterion_results: Annotated[list[CriterionResult], append_results]

    total_score: float
    verification: VerificationResult

    regrade_count: int
    final_report: FinalReport

## 9. LangGraph nodes

Each node has one responsibility.

The graph is deliberately more explicit than a single LLM prompt so that each stage can be inspected, tested, and replaced independently.

In [ ]:
def evaluate_criterion(state: dict):
    criterion: Criterion = state["criterion"]
    evaluator = llm.with_structured_output(CriterionResult)

    prompt = f"""
You are grading ONE criterion of a student's answer.

Create an auditable mapping:
RUBRIC REQUIREMENT -> EXACT STUDENT ANSWER PART -> MARKS.

Question:
{state["question"]}

Reference answer:
{state["reference_answer"]}

Student plaintext answer:
-------------------------
{state["student_answer"]}
-------------------------

Answer analysis:
{state["answer_analysis"].model_dump_json(indent=2)}

Criterion:
{criterion.model_dump_json(indent=2)}

Global grading rules:
{state["rubric"].global_rules}

Instructions:
1. Examine the student's answer literally.
2. Identify the exact sentence(s), clause(s), phrase(s), equation(s), or
   other part(s) that represent this criterion.
3. Copy those answer parts VERBATIM into supporting_answer_parts.
4. Do not paraphrase the student's evidence.
5. State which rubric requirement each selected answer part represents.
6. If no relevant evidence exists, use [] and evidence_status="absent".
7. If the evidence is wrong, use evidence_status="incorrect".
8. If only part is satisfied, use evidence_status="partial".
9. Do not award marks merely because a keyword appears.
10. Do not infer knowledge that the student did not express.
11. Do not rewrite incorrect student statements into correct statements.
12. Never exceed {criterion.max_marks} marks.
13. Keep missing_requirements explicit.

Return a complete CriterionResult.
"""

    result = evaluator.invoke(prompt)

    result.awarded_marks = max(
        0.0,
        min(float(result.awarded_marks), float(criterion.max_marks))
    )
    result.maximum_marks = float(criterion.max_marks)
    result.criterion_id = criterion.id
    result.criterion_name = criterion.name
    result.evidence.criterion_id = criterion.id
    result.evidence.criterion_name = criterion.name

    return {"criterion_results": [result]}


## 10. Build the LangGraph

Notice the conditional path after verification.

A failed verification can trigger one bounded regrade before the graph terminates.

In [ ]:
builder = StateGraph(GradingState)

builder.add_node("load_rules", load_rules)
builder.add_node("parse_rubric", parse_rubric)
builder.add_node("analyze_answer", analyze_answer)
builder.add_node("evaluate_criterion", evaluate_criterion)
builder.add_node("aggregate_score", aggregate_score)
builder.add_node("verify_grade", verify_grade)
builder.add_node("regrade", regrade)
builder.add_node("final_report", final_report)

builder.add_edge(START, "load_rules")
builder.add_edge("load_rules", "parse_rubric")
builder.add_edge("parse_rubric", "analyze_answer")

# Dynamic fan-out.
builder.add_conditional_edges(
    "analyze_answer",
    fan_out_criteria,
    ["evaluate_criterion"],
)

# All criterion branches converge here.
builder.add_edge("evaluate_criterion", "aggregate_score")
builder.add_edge("aggregate_score", "verify_grade")

builder.add_conditional_edges(
    "verify_grade",
    route_after_verification,
    {
        "regrade": "regrade",
        "final_report": "final_report",
    },
)

builder.add_edge("regrade", "aggregate_score")
builder.add_edge("final_report", END)

graph = builder.compile()

print("Graph compiled successfully.")

## 11. Run the graph

The input contains only the question, reference answer, and student answer.

The graph itself loads `grading_rules.txt`.

In [ ]:
initial_state: GradingState = {
    "question": question,
    "reference_answer": reference_answer,
    "student_answer": student_answer,
    "regrade_count": 0,
}

result = graph.invoke(initial_state)

report = result["final_report"]

print(f"Score: {report.total_score:.1f}/{report.maximum_score:.1f}")
print(f"Percentage: {report.percentage:.1f}%")
print()
print("Overall feedback:")
print(report.overall_feedback)

## 12. Display criterion-level grading

In [ ]:
for r in report.criterion_results:
    print(f"[{r.criterion_id}] {r.criterion_name}")
    print(f"Score: {r.awarded_marks}/{r.maximum_marks}")
    print("Evidence:")
    for evidence in r.evidence:
        print("  -", evidence)
    if r.missing_requirements:
        print("Missing:")
        for item in r.missing_requirements:
            print("  -", item)
    print("Reasoning:", r.reasoning)
    print("-" * 70)

## 13. Inspect verification

The verification result is useful for auditing why a grade was accepted or regraded.

In [ ]:
verification = result["verification"]

print("Valid:", verification.valid)
print("Requires regrading:", verification.requires_regrading)

if verification.issues:
    print("\nIssues:")
    for issue in verification.issues:
        print("-", issue)
else:
    print("\nNo verification issues reported.")

print("\nRegrade count:", result.get("regrade_count", 0))

## 14. Export an audit-friendly JSON report

The output keeps the rubric, answer analysis, criterion evidence, score, verification result, and final feedback together.

This is useful if the project later becomes a web service or batch-grading pipeline.

In [ ]:
audit_record = {
    "question": question,
    "reference_answer": reference_answer,
    "student_answer": student_answer,
    "rubric": result["rubric"].model_dump(),
    "answer_analysis": result["answer_analysis"].model_dump(),
    "criterion_results": [
        r.model_dump()
        for r in result["final_report"].criterion_results
    ],
    "total_score": result["final_report"].total_score,
    "maximum_score": result["final_report"].maximum_score,
    "percentage": result["final_report"].percentage,
    "verification": result["verification"].model_dump(),
    "regrade_count": result.get("regrade_count", 0),
    "overall_feedback": result["final_report"].overall_feedback,
}

Path("grading_result.json").write_text(
    json.dumps(audit_record, indent=2),
    encoding="utf-8",
)

print("Saved:", Path("grading_result.json").resolve())

## 15. Test a different student answer without changing the graph

This demonstrates the main design goal: grading behavior comes from the external rubric.

Only replace the answer.

In [ ]:
student_answer_2 = '''
A process is an executing program and normally has its own address space.
A thread is an execution path inside a process. Threads in one process share
the address space and resources such as code and data. Switching between
threads is usually cheaper than switching between separate processes.
'''

result_2 = graph.invoke({
    "question": question,
    "reference_answer": reference_answer,
    "student_answer": student_answer_2,
    "regrade_count": 0,
})

report_2 = result_2["final_report"]

print(f"Score: {report_2.total_score:.1f}/{report_2.maximum_score:.1f}")
print(f"Percentage: {report_2.percentage:.1f}%")
print(report_2.overall_feedback)

## 16. Change the grading policy without changing Python code

For example, you can create a new policy file for another question.

The same graph can then ingest it by changing only the filename in `load_rules`.

A production implementation should pass the rules path through graph state rather than hard-coding it.

A useful next step is to change:

```python
rules_path: str
```

into the graph state and make `load_rules` read that path.

This allows:

```text
rules/
├── operating_systems.txt
├── algorithms.txt
├── computer_networks.txt
├── databases.txt
└── machine_learning.txt
```

with the same grading engine.

## 17. Production improvements

This notebook is intentionally self-contained. For a portfolio-grade implementation, add:

### Reliability
- calibrated confidence scores
- teacher-vs-LLM benchmark datasets
- inter-rater agreement metrics
- adversarial answer tests
- prompt-injection tests
- regression tests for rubric changes

### Architecture
- persistent LangGraph checkpoints
- human-in-the-loop review using `interrupt`
- batch grading
- async criterion evaluation
- model fallback
- retry/error handling
- structured logging

### Inputs
- PDF/DOCX answer ingestion
- OCR for handwritten answers
- multiple questions per submission
- answer normalization

### Evaluation
Compare:
1. single-pass LLM grading
2. criterion-level LangGraph grading
3. criterion-level grading + verification
4. human-reviewed grading

Useful metrics include:
- Mean Absolute Error
- exact score agreement
- ±1-mark agreement
- Pearson correlation
- Spearman correlation
- criterion-level agreement

### Security
Treat the student answer and even the rubric as untrusted input. Do not let text inside an answer override system-level grading instructions.

### Important design principle

The LLM should **propose and explain grading decisions**.

Python should enforce:
- maximum marks
- score arithmetic
- schema validity
- retry limits
- allowed graph transitions

That separation makes the system much easier to audit.

## Evidence-aware plaintext grading report

For every marking rubric, the report lists:
- the rubric requirements,
- the exact sentence(s)/part(s) in the student's answer that represent it,
- which requirement each part represents,
- evidence status,
- missing requirements,
- awarded marks,
- grading reasoning,
- independent verification issues.

The evidence excerpts are copied from the normalized plaintext answer and are
intended for later verification against the original image/raw OCR.


In [ ]:
def build_text_report(
    result: GradingState,
    source_image: str | None = None,
) -> str:
    report = result["final_report"]
    rubric = result["rubric"]
    verification = result["verification"]

    lines = [
        "=" * 90,
        "RUBRICGRAPH GRADING REPORT",
        "=" * 90,
    ]

    if source_image:
        lines.append(f"Source image: {source_image}")

    lines.extend([
        f"Question: {result['question']}",
        f"Final score: {report.total_score:.1f}/{report.maximum_score:.1f} "
        f"({report.percentage:.1f}%)",
        "",
        "STUDENT PLAINTEXT ANSWER",
        "-" * 90,
        result["student_answer"],
        "",
        "RUBRIC-BY-RUBRIC EVIDENCE",
        "=" * 90,
    ])

    for criterion in rubric.criteria:
        r = next(
            x for x in report.criterion_results
            if x.criterion_id == criterion.id
        )

        lines.extend([
            "",
            f"RUBRIC {criterion.id}: {criterion.name}",
            f"MARKS: {r.awarded_marks:.1f}/{r.maximum_marks:.1f}",
            f"EVIDENCE STATUS: {r.evidence.evidence_status.upper()}",
            "",
            "RUBRIC REQUIREMENTS:",
        ])

        for req in criterion.requirements:
            lines.append(f"  - {req}")

        lines.extend([
            "",
            "STUDENT ANSWER SENTENCE(S)/PART(S) REPRESENTING THIS RUBRIC:",
        ])

        if r.evidence.supporting_answer_parts:
            for part in r.evidence.supporting_answer_parts:
                lines.append(f'  "{part}"')
        else:
            lines.append("  [NONE]")

        lines.extend(["", "REPRESENTED REQUIREMENTS:"])

        if r.evidence.represented_requirements:
            for req in r.evidence.represented_requirements:
                lines.append(f"  - {req}")
        else:
            lines.append("  [NONE]")

        lines.extend(["", "MISSING REQUIREMENTS:"])

        if r.missing_requirements:
            for req in r.missing_requirements:
                lines.append(f"  - {req}")
        else:
            lines.append("  [NONE]")

        lines.extend([
            "",
            "EVIDENCE EXPLANATION:",
            r.evidence.explanation,
            "",
            "GRADING REASONING:",
            r.reasoning,
            "-" * 90,
        ])

    lines.extend([
        "",
        "VERIFICATION",
        "=" * 90,
        f"Valid: {verification.valid}",
        f"Regrading performed: {result.get('regrade_count', 0) > 0}",
    ])

    if verification.issues:
        lines.append("")
        lines.append("Verifier issues:")
        for issue in verification.issues:
            lines.append(f"  - {issue}")
    else:
        lines.append("No verifier issues reported.")

    lines.extend([
        "",
        "OVERALL FEEDBACK",
        "-" * 90,
        report.overall_feedback,
    ])

    return "\n".join(lines)


text_report = build_text_report(result)

Path("grading_report.txt").write_text(
    text_report,
    encoding="utf-8",
)

print(text_report)
print("\nSaved to:", Path("grading_report.txt").resolve())


## Complete evidence-aware audit package

For a handwritten submission, preserve the following:

```text
submission/
├── original.jpg
├── raw_transcription.txt
├── student_answer.txt
├── ocr_metadata.json
├── grading_report.txt
└── grading_result.json
```

This creates a traceable chain:

```text
image
  -> raw OCR
  -> normalized plaintext
  -> rubric
  -> exact answer evidence
  -> criterion score
  -> verification
  -> final score
```


In [ ]:
def save_evidence_audit(
    result: GradingState,
    output_dir: str = "submission",
    source_image: str | None = None,
    ocr_result: OCRResult | None = None,
):
    output = Path(output_dir)
    output.mkdir(parents=True, exist_ok=True)

    (output / "student_answer.txt").write_text(
        result["student_answer"],
        encoding="utf-8",
    )

    if ocr_result is not None:
        (output / "raw_transcription.txt").write_text(
            ocr_result.raw_transcription,
            encoding="utf-8",
        )
        (output / "ocr_metadata.json").write_text(
            json.dumps(
                {
                    "source_image": source_image,
                    "confidence": ocr_result.confidence,
                    "uncertain_segments": ocr_result.uncertain_segments,
                },
                indent=2,
            ),
            encoding="utf-8",
        )

    report_text = build_text_report(result, source_image)
    (output / "grading_report.txt").write_text(
        report_text,
        encoding="utf-8",
    )

    audit = {
        "source_image": source_image,
        "question": result["question"],
        "reference_answer": result["reference_answer"],
        "student_answer": result["student_answer"],
        "rubric": result["rubric"].model_dump(),
        "answer_analysis": result["answer_analysis"].model_dump(),
        "criterion_results": [
            r.model_dump()
            for r in result["final_report"].criterion_results
        ],
        "total_score": result["final_report"].total_score,
        "maximum_score": result["final_report"].maximum_score,
        "percentage": result["final_report"].percentage,
        "verification": result["verification"].model_dump(),
        "regrade_count": result.get("regrade_count", 0),
        "overall_feedback": result["final_report"].overall_feedback,
    }

    (output / "grading_result.json").write_text(
        json.dumps(audit, indent=2),
        encoding="utf-8",
    )

    return output


## End-to-end handwritten grading

The following function performs OCR first, then sends the resulting plaintext
answer through the LangGraph. It saves the exact evidence mappings for every
rubric item.

If OCR confidence is below the threshold, grading is stopped for human review.


In [ ]:
def grade_handwritten_answer(
    image_path: str,
    question: str,
    reference_answer: str,
    rules_path: str = "grading_rules.txt",
    output_dir: str = "submission",
    min_ocr_confidence: float = 0.80,
):
    ocr_result = ocr_handwritten_image(image_path)

    if ocr_result.confidence < min_ocr_confidence:
        output = Path(output_dir)
        output.mkdir(parents=True, exist_ok=True)

        (output / "raw_transcription.txt").write_text(
            ocr_result.raw_transcription,
            encoding="utf-8",
        )
        (output / "student_answer.txt").write_text(
            ocr_result.plaintext_answer,
            encoding="utf-8",
        )
        (output / "ocr_metadata.json").write_text(
            json.dumps(
                {
                    "source_image": image_path,
                    "confidence": ocr_result.confidence,
                    "uncertain_segments": ocr_result.uncertain_segments,
                    "human_review_required": True,
                },
                indent=2,
            ),
            encoding="utf-8",
        )

        return {
            "status": "human_review_required",
            "ocr": ocr_result.model_dump(),
            "reason": (
                f"OCR confidence {ocr_result.confidence:.2f} is below "
                f"threshold {min_ocr_confidence:.2f}."
            ),
        }

    graph_result = graph.invoke({
        "rules_path": rules_path,
        "question": question,
        "reference_answer": reference_answer,
        "student_answer": ocr_result.plaintext_answer,
        "regrade_count": 0,
    })

    save_evidence_audit(
        graph_result,
        output_dir=output_dir,
        source_image=image_path,
        ocr_result=ocr_result,
    )

    return {
        "status": "graded",
        "ocr": ocr_result.model_dump(),
        "grading": graph_result["final_report"].model_dump(),
        "verification": graph_result["verification"].model_dump(),
        "regrade_count": graph_result.get("regrade_count", 0),
        "report_path": str(Path(output_dir, "grading_report.txt")),
    }
